In [ ]:
def write_runscript_from_config(config_data: dict, output_path: str):
    """
    根据配置字典生成一个类似 runscript_test 的固定格式运行文件。
    该版本能处理值为列表（每场景不同）或字符串（所有场景通用）的参数。

    参数:
    config_data (dict): 包含所有模拟参数的字典。
    output_path (str): 输出文件的路径。
    """
    lines = []
    
    # --- 1. 提取通用设置 ---
    general_config = config_data.get("SETUP_GENERAL", {})

    print(general_config)
    
    grid_line = ' '.join(map(str, general_config.get("grid_dims", [1, 1, 1, 1, 1])))
    lines.append(grid_line)

    lines.append(general_config.get("site_data_file"))
    lines.append(general_config.get("topography_data_file"))
    
    num_scenes = general_config.get("num_scenes", 0)
    num_runs = general_config.get("num_runs", 1)
    lines.append(f"{num_scenes} {num_runs}")
    
    # --- 2. 循环处理每个场景 ---
    scenes_config = config_data.get("SETUP_SCENES", {})
    # 定义场景中文件参数的顺序
    scene_file_keys = [
        "weather_data_files", "weather_options_files", "land_management_files",
        "plant_management_files", "soil_output_1", "atmospheric_output",
        "n_flux_output", "p_flux_output", "soil_output_2", "soil_output_3",
        "water_props_output", "n_props_output", "p_props_output", "t_props_output"
    ]

    for i in range(num_scenes):
        lines.append("1 1")
        
        for key in scene_file_keys:
            config_value = scenes_config.get(key)
            
            # **智能处理逻辑**
            # 如果值是列表，则按场景索引取值
            if isinstance(config_value, list):
                if i < len(config_value):
                    lines.append(config_value[i])
                else:
                    lines.append("NO_FILE_SPECIFIED_IN_LIST")
            # 如果值是字符串，则所有场景都使用该值
            elif isinstance(config_value, str):
                lines.append(config_value)
            # 如果未提供值
            else:
                lines.append("NO_FILE_SPECIFIED")

    # --- 3. 添加结束标志 ---
    lines.append("0 0")

    # --- 4. 将所有行写入文件 ---
    try:
        with open(output_path, 'w') as f:
            f.write('\n'.join(lines))
        print(f"文件已成功生成在: {output_path}")
    except IOError as e:
        print(f"写入文件时出错: {e}")


# =======================================================================
#                           --- 使用示例 ---
# =======================================================================

if __name__ == '__main__':
    # 1. 定义新的配置字典，注意值的变化
    my_config = {
        "SETUP_GENERAL": {
            "grid_dims": [1, 1, 1, 1, 1],
            "site_data_file": "st022852.txt",
            "topography_data_file": "tp022852",
            "num_scenes": 3,
            "num_runs": 1
        },
        "SETUP_SCENES": {
            # 这个参数保持为列表，因为每个场景的气象数据通常不同
            "weather_data_files":     ['w1980022852', 'w1981022852', 'w1982022852'], 
            # --- 以下参数已从列表改为单个字符串 ---
            "weather_options_files":  ['opt1800', 'opt1801', 'opt1802'],
            "land_management_files":  'sm',
            "plant_management_files": 'pft_arctic_g',
            "soil_output_1":          'NO',
            "atmospheric_output":     'NO',
            "n_flux_output":          'NO',
            "p_flux_output":          'NO',
            "soil_output_2":          'NO',
            "soil_output_3":          'dc',
            "water_props_output":     'dw',
            "n_props_output":         'dn',
            "p_props_output":         'NO',
            "t_props_output":         'dh'
        }
    }

    # 2. 调用函数，指定输出文件名
    output_filename = "runscript_generated_single_element.txt"
    write_runscript_from_config(my_config, output_filename)

In [ ]:
import collections.abc

def write_sitedata_from_config(config_data: dict, output_path: str):
    """
    根据配置字典生成一个 site_data 格式的文件 (如 st022852)。

    参数:
    config_data (dict): 包含所有站点参数的字典。
    output_path (str): 输出文件的路径。
    """
    lines = []
    params = config_data.get("SITE_PARAMETERS", {})

    # --- Line 1: 地理和基本水文信息 ---
    line1 = (f"{params.get('latitude', 0.0)} {params.get('altitude', 0.0)} "
             f"{params.get('mean_temp_c', 0.0)} {params.get('water_table_flag', 0.0)}")
    lines.append(line1)

    # --- Line 2: 大气组分 ---
    # O2, N2, CO2, CH4, N2O, NH3
    atm_comp = params.get('atm_composition_ppm', [])
    lines.append(' '.join(map(str, atm_comp)))

    # --- Line 3: 气候、网格和水文参数 ---
    # 柯本气候区, 盐分, 侵蚀选项(0=冻融), 网格连接(1=行),自然/人工地下水深, 地下水坡度
    # 
    cgh_params = params.get('climate_grid_hydro_params', [])
    lines.append(' '.join(map(str, cgh_params)))

    # --- Line 4: 边界条件 (合并多个参数) ---
    bc_surf = params.get('bc_surface_runoff_nesw', [0.0]*4) # N E S W 边界条件 (地表径流)
    bc_sub = params.get('bc_subsurface_flow_nesw', [0.0]*4) # N E S W 边界条件 (地下径流)
    dist_wt = params.get('dist_water_table_nesw', [0.0]*4) # N E S W 到地下水表的距离 (m)
    lower_bc = params.get('lower_bc_water_flow', 0.0)
    
    # 确保 lower_bc 是一个可迭代对象以便拼接
    if not isinstance(lower_bc, collections.abc.Iterable):
        lower_bc = [lower_bc]

    full_bc_line_values = list(bc_surf) + list(bc_sub) + list(dist_wt) + list(lower_bc)
    lines.append(' '.join(map(str, full_bc_line_values)))

    # --- Line 5: 东西向宽度 ---
    lines.append(str(params.get('width_we_column', 1.0)))
    
    # --- Line 6: 南北向宽度 ---
    lines.append(str(params.get('width_ns_row', 1.0)))

    # --- 写入文件 ---
    try:
        with open(output_path, 'w') as f:
            f.write('\n'.join(lines))
        print(f"文件已成功生成在: {output_path}")
    except IOError as e:
        print(f"写入文件时出错: {e}")

# =======================================================================
#                           --- 使用示例 ---
# =======================================================================

if __name__ == '__main__':
    # 1. 定义与 Namelist 结构一致的Python字典
    my_site_config = {
        "SITE_PARAMETERS": {
            "latitude": 69.1,
            "altitude": 130.0,
            "mean_temp_c": -8.0,
            "water_table_flag": 1.0, # 地下水表标记 (1 = 自然静止)
            "atm_composition_ppm": [210000.0, 780000.0, 282.9, 1.8, 0.3, 0.005], # O2, N2, CO2, CH4, N2O, NH3
            "climate_grid_hydro_params": [62, 0, -1, 1, 3, 100.0, 0.0], # 柯本气候区, 盐分, 侵蚀选项(0=冻融), 网格连接(1=行),自然地下水深, 人工地下水深, 地下水坡度
            "bc_surface_runoff_nesw": [0.0, 0.0, 0.0, 0.0], # N E S W 边界条件 (地表径流)
            "bc_subsurface_flow_nesw": [0.0, 0.0, 0.0, 0.0], # N E S W 边界条件 (地下径流)
            "dist_water_table_nesw": [0.0, 0.0, 0.0, 0.0], # N E S W 到地下水表的距离 (m)
            "lower_bc_water_flow": 0.0, # 水流的下边界条件
            "width_we_column": 1.0,
            "width_ns_row": 1.0
        }
    }

    # 2. 调用函数，指定输出文件名
    output_filename = "st022852.txt"
    write_sitedata_from_config(my_site_config, output_filename)